In [1]:
# --- Бібліотеки для даної роботи ---
try:
    import numpy, pandas, matplotlib, plotly, sklearn, jupyterlab, ipywidgets
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install -q numpy pandas matplotlib plotly scikit-learn "jupyterlab>=3" "ipywidgets>=7.6"

Бібліотеки вже встановлені. Пропускаємо інсталяцію.


## Домашнє завдання: Тема 10. EM-алгоритм та розділення суміші Гаусівських функцій

### **Це допоможе закріпити такі навички:**

- Попередньої підготовки даних до моделювання
- Роботу з бібліотеками аналізу даних

### **Завдання (крок за кроком):**

***Для цієї задачі необхідно буде завантажити дані [World Happiness Report](https://www.kaggle.com/datasets/unsdsn/world-happiness).***

Для виконання завдання необхідно виконати такі кроки:

1. **Інсталювати та імпортувати необхідні бібліотеки:** 
    - Необхідно буде інсталювати такі пакети:
	```bash
	!pip install plotly==5.20.0
	!pip install "jupyterlab>=3" "ipywidgets>=7.6"
	```

2. **Завантажити дані:**
    - З набору https://www.kaggle.com/datasets/unsdsn/world-happiness.
	```bash
	!wget -O WorldHappinessReport.zip https://github.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/blob/main/WorldHappinessReport.zip?raw=true
	```

3. **Розпакувати дані:**
    ```bash
	!unzip WorldHappinessReport.zip
	```

4. **Прочитати дані та відобразити загальну інформацію про:**
	- Статистики
	- Типи ознак

5. **Побудувати діаграми розподілу числових ознак:**
    - Проаналізувати на відповідність чи не відповідність нормальному розподілу.

6. **Відібрати числових ознак та кореляційну матрицю:**
    - Виходячи із розуміння домену та даних відібрати певну кількість числових ознак
    - Відобразити кореляційну матрицю (*див. Тема 4. Вимірювання відстаней та подібностей в аналізі даних*)

7. **Зробити висновок про:**
    - Наявність та силу лінійного зв'язку між ознаками.

8. **Відобразити розподіл:**
    - Цільової ознаки (Happiness.Score або Happiness.Rank) за країнами.
    - Використовуючи наведений нижче код для побудови теплової мапи.
	```py
	fig = px.choropleth(data_dataframe,
						locations = "Country",
						color = "Happiness.Score",
						locationmode = "country names",
                    	)
	fig.update_layout(title = "Happiness Index 2017")
	fig.show()
	```

9. **Застосувати стандартизацію даних:**
    - Для приведення всіх значень до одного діапазону статистик.
    - Використовуючи функцію data_scale() та наступні перетворення
	```py
	def data_scale(data, scaler_type='minmax'):
	    from sklearn.preprocessing import MinMaxScaler
	    from sklearn.preprocessing import StandardScaler
	    from sklearn.preprocessing import Normalizer
	    if scaler_type == 'minmax':
	        scaler = MinMaxScaler()
	    if scaler_type == 'std':
	        scaler = StandardScaler()
	    if scaler_type == 'norm':
	        scaler = Normalizer()

	    scaler.fit(data)
	    res = scaler.transform(data)
	    return res

	data_scaled = data_scale(original_dataframe)
	df_scaled = pd.DataFrame(data_scaled, columns=[original_dataframe.columns])
	print(df_scaled.head())
	```

10. **Відобразити статистики:**
    - Отриманого стандартизованого набору даних та порівняти зі статистиками оригінального набору даних.
    - Зробити висновки.

11. **Побудувати модель кластеризації:**
	- Засобами функції `GaussianMixture()` бібліотеки `sklearn`.

12. **Побудувати теплову мапу:**
    - Для відображення розподілу країн за кластерами.

13. **Дослідити вплив:**
    - Різного набору ознак
    - Результат кластеризації

14. **Висновок:**
    - Зробити загальний висновок про відповідність результатів кластеризації оригінальному розподілу країн за ознакою.

**1. Імпорт необхідних бібліотек:**

In [45]:
# 1. СТАНДАРТНІ БІБЛІОТЕКИ PYTHON (Мережа, Файлова система, Попередження)
import os
import shutil
import urllib.request
import zipfile
import warnings

warnings.filterwarnings('ignore')

# 2. РОБОТА З ДАНИМИ ТА МАТЕМАТИКА
import math
import numpy as np
import pandas as pd

# 3. МАШИННЕ НАВЧАННЯ (Кластеризація та Препроцесинг)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer

# 4. MLOps ТА СЕРІАЛІЗАЦІЯ МОДЕЛЕЙ
import joblib

# 5. ВІЗУАЛІЗАЦІЯ ТА UI (Plotly, IPywidgets & HTML)
from IPython.display import HTML, display
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats

print("📦 Модулі архітектури імпортовано успішно!")

📦 Модулі архітектури імпортовано успішно!


**1.3. Конфігурація експерименту (Глобальні змінні):**

In [41]:
# 1. МЕРЕЖА ТА ФАЙЛОВА СИСТЕМА
DATA_DIR                = "WorldHappinessDataSet"                               # Папка для ізольованого збереження всіх сирих даних
DATASET_URL             = "https://www.kaggle.com/api/v1/datasets/download/unsdsn/world-happiness"
ZIP_PATH                = os.path.join(DATA_DIR, "world-happiness.zip")

TARGET_YEAR             = "2017"                                                # Доступні роки: "2015" | "2016" | "2017" | "2018" | "2019"
CSV_FILENAME            = os.path.join(DATA_DIR, f"{TARGET_YEAR}.csv")          # Динамічний шлях до потрібного файлу

# 2. СТРУКТУРА ДАНИХ ТА ОЗНАКИ (Вирішення проблеми Schema Drift)
SCHEMA_MAPPING = {
    "2015": {
        "country": "Country",
        "target": "Happiness Score",
        "drop": ["Happiness Rank", "Standard Error", "Region"],
        "features": ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)", "Freedom", "Trust (Government Corruption)"]
    },
    "2016": {
        "country": "Country",
        "target": "Happiness Score",
        "drop": ["Happiness Rank", "Lower Confidence Interval", "Upper Confidence Interval", "Region"],
        "features": ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)", "Freedom", "Trust (Government Corruption)"]
    },
    "2017": {
        "country": "Country",
        "target": "Happiness.Score",
        "drop": ["Happiness.Rank", "Whisker.high", "Whisker.low"],
        "features": ["Economy..GDP.per.Capita.", "Family", "Health..Life.Expectancy.", "Freedom", "Trust..Government.Corruption."]
    },
    "2018": {
        "country": "Country or region",
        "target": "Score",
        "drop": ["Overall rank"],
        "features": ["GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    },
    "2019": {
        "country": "Country or region",
        "target": "Score",
        "drop": ["Overall rank"],
        "features": ["GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    }
}

CURRENT_SCHEMA          = SCHEMA_MAPPING[TARGET_YEAR]
COUNTRY_COL             = CURRENT_SCHEMA["country"]             # Динамічна колонка країни (змінювалась у 2018)
TARGET_METRIC           = CURRENT_SCHEMA["target"]              # Головна цільова метрика (Індекс щастя)
DROP_COLUMNS            = CURRENT_SCHEMA["drop"]                # Технічні колонки, що не несуть користі для кластеризації
FEATURES_FULL           = CURRENT_SCHEMA["features"]            # Повний набір соціально-економічних ознак для GMM
FEATURES_MINI           = [FEATURES_FULL[0], FEATURES_FULL[2]]  # Зменшений набір (ВВП та Здоров'я) для дослідження розмірності

# 3. МАШИННЕ НАВЧАННЯ (GMM) ТА СЕРІАЛІЗАЦІЯ
N_CLUSTERS              = 3                                     # Кількість кластерів: задає число прихованих Гаусівських розподілів (Високий, Середній, Низький рівень)
COVARIANCE_TYPE         = 'full'                                # Форма матриці коваріації (геометрія кластерів): 'full' - різні еліпси під будь-яким кутом | 'tied' - однакова форма та нахил для всіх | 'diag' - еліпси строго паралельні осям координат | 'spherical' - ідеальні круглі сфери різного радіусу
GMM_INIT_PARAMS         = 'kmeans'                              # Стратегія стартової ініціалізації (Крок 0 для EM): 'kmeans' - розумний розвідник для надійного старту | 'random' - повністю випадкові координати в просторі | 'random_from_data' - випадкові реальні точки з набору даних
SCALER_TYPE             = 'std'                                 # Алгоритм масштабування простору ознак: 'std' - центрує дисперсію навколо нуля (ідеально для GMM) | 'minmax' - жорстко стискає дані в межі від 0 до 1 | 'norm' - нормує самі вектори по їхній абсолютній довжині
N_INIT                  = 10                                    # Кількість перезапусків EM-алгоритму: захист від застрягання моделі в поганих локальних мінімумах
RANDOM_STATE            = 42                                    # Фіксація генератора псевдовипадкових чисел: гарантує 100% відтворюваність результатів експерименту

MODEL_DIR               = "GMM_Models"                          # Папка для збереження серіалізованих об'єктів
MODEL_PATH              = os.path.join(MODEL_DIR, f"gmm_{TARGET_YEAR}_{COVARIANCE_TYPE}_{GMM_INIT_PARAMS}_model.pkl") # Динамічне ім'я моделі
SCALER_PATH             = os.path.join(MODEL_DIR, f"scaler_{TARGET_YEAR}_{SCALER_TYPE}.pkl")                          # Динамічне ім'я скейлера

# 4. ВІЗУАЛІЗАЦІЯ ТА UI
PLOT_TEMPLATE           = "plotly_dark"                         # Темна тема для інтерактивних графіків Plotly
MAP_LOCATION_MODE       = "country names"                       # Режим розпізнавання країн для мап Choropleth
COLOR_SCALE_HAPPINESS   = "Viridis"                             # Безперервний градієнт для оригінального індексу щастя
COLOR_PALETTE_FULL      = px.colors.qualitative.Set1            # Контрастні дискретні кольори для 3-х кластерів (повний набір)
COLOR_PALETTE_MINI      = px.colors.qualitative.Pastel          # Пастельні кольори для експерименту зі зменшеною розмірністю

TABLE_PROPS             = {'background-color': '#1e1e1e', 'color': '#00c3ff', 'border': '1px solid #444', 'text-align': 'center'}
DESCRIBE_CMAP           = 'YlGn'                                # Кольорова схема (Yellow-Green) для підсвічування описових статистик

print(f"⚙️ Глобальні константи ініціалізовано!\n   Рік: {TARGET_YEAR} | GMM({COVARIANCE_TYPE}, {GMM_INIT_PARAMS}) + {SCALER_TYPE} Scaler")

⚙️ Глобальні константи ініціалізовано!
   Рік: 2017 | GMM(full, kmeans) + std Scaler


**1.7. Приклад на HTML (Анатомія GMM):**

In [10]:
C_RAW = "#888888"                               # Базовий колір для "нерозмічених" (сирих) даних у просторі
C_C1  = COLOR_PALETTE_FULL[0]                   # Динамічний колір Кластера 1 (підтягується з глобальної палітри констант)
C_C2  = COLOR_PALETTE_FULL[1]                   # Динамічний колір Кластера 2
C_C3  = COLOR_PALETTE_FULL[2]                   # Динамічний колір Кластера 3

html_em_pipeline = f"""
<div style="font-family: sans-serif; max-width: 900px; background-color: #111; padding: 20px; border-radius: 10px; border: 1px solid #333; margin: auto;">
    <h2 style="color: #00c3ff; text-align: center; margin-top: 0;">🧠 Анатомія GMM: Що робить EM-алгоритм з країнами?</h2>
    
    <div style="background-color: #1a1a1a; padding: 15px; margin-bottom: 15px; border-left: 5px solid {C_RAW}; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 0: Сирий простір (Дані після {SCALER_TYPE} Scaler)</div>
        <div style="color: {C_RAW}; font-size: 15px; margin-top: 5px; font-style: italic;">
            Маємо N країн у багатовимірному просторі ознак (ВВП, Здоров'я, Свобода...).<br>
            Усі точки "сірі", алгоритм ще нічого не знає про кластери.
        </div>
    </div>

    <div style="text-align: center; color: #ffd700; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ffd700; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 1: Ініціалізація (Метод '{GMM_INIT_PARAMS}')</div>
        <div style="color: #ffd700; font-size: 15px; margin-top: 5px;">
            ШІ генерує {N_CLUSTERS} випадкові багатовимірні "дзвони" (Гаусівські розподіли).<br>
            Кожен має свій центр <b>(μ)</b> та матрицю коваріації <b>(Σ)</b>.
        </div>
    </div>

    <div style="text-align: center; color: #ff9900; font-size: 20px;">⬇ ♻️ Цикл EM-алгоритму ♻️ ⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ff9900; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 2: E-крок (Expectation / Очікування)</div>
        <div style="color: #ff9900; font-size: 15px; margin-top: 5px;">
            Обчислення м'якої ймовірності (Soft Clustering) за формулою Баєса:<br>
            <i>"Країна Х належить до Кластера-1 на 10%, Кластера-2 на 85%, Кластера-3 на 5%".</i>
        </div>
    </div>

    <div style="text-align: center; color: #00aaff; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #00aaff; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 3: M-крок (Maximization / Максимізація)</div>
        <div style="color: #00aaff; font-size: 15px; margin-top: 5px;">
            Оновлення параметрів дзвонів: <b>Нові μ</b> тягнуться до скупчень точок, <b>Нові Σ</b> змінюють форму еліпсів.
        </div>
    </div>

    <div style="text-align: center; color: #00ffcc; font-size: 20px; margin-top: 10px;">⬇</div>

    <div style="background-color: #222; padding: 15px; margin-top: 15px; border: 2px dashed #00ffcc; border-radius: 5px; text-align: center;">
        <div style="color: #888; font-size: 14px; font-weight: bold; text-transform: uppercase;">✓ Фінал: Збіжність (Convergence)</div>
        <div style="color: #00ffcc; font-size: 18px; margin-top: 10px; font-family: monospace;">[ <span style="color:{C_C1}">Кластер 1</span> | <span style="color:{C_C2}">Кластер 2</span> | <span style="color:{C_C3}">Кластер 3</span> ]</div>
    </div>
</div>
"""

print("Красивий Вивід (Інтерактивна схема логіки алгоритму):")
display(HTML(html_em_pipeline))

Красивий Вивід (Інтерактивна схема логіки алгоритму):


**2. Завантажити дані:**

In [36]:
def is_valid_zip(filepath):
    if not os.path.exists(filepath) or not zipfile.is_zipfile(filepath):
        return False
    try:
        with zipfile.ZipFile(filepath, 'r') as z:
            if z.testzip() is not None:
                return False
    except Exception:
        return False
    return True

def download_dataset():
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"⏳ Завантаження архіву у папку '{DATA_DIR}'...")
    try:
        urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
        print("✅ Завантаження завершено.")
    except Exception as e:
        print(f"❌ Мережева помилка завантаження: {e}")

if os.path.exists(ZIP_PATH):
    print("🔍 Перевірка цілісності існуючого архіву...")
    if not is_valid_zip(ZIP_PATH):
        print("🪫 Архів пошкоджено. Видаляємо та завантажуємо наново...")
        os.remove(ZIP_PATH)
        download_dataset()
    else:
        print("🔋 Архів цілий. Пропускаємо мережевий запит.")
else:
    download_dataset()

⏳ Завантаження архіву у папку 'WorldHappinessDataSet'...
✅ Завантаження завершено.


**3. Розпакувати дані:**

In [43]:
if os.path.exists(CSV_FILENAME):
    print(f"⚡ Файл '{CSV_FILENAME}' вже розпаковано та готовий до роботи.")
elif os.path.exists(ZIP_PATH):
    if is_valid_zip(ZIP_PATH):
        print(f"📦 Аналізуємо вміст архіву '{ZIP_PATH}'...")
        try:
            with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
                csv_files = [f for f in zip_ref.namelist() if f.endswith('.csv')]
                if not csv_files:
                    raise Exception("В архіві немає CSV файлів!")

                expected_file_suffix = f"{TARGET_YEAR}.csv"
                target_csv_in_zip = next((f for f in csv_files if f.endswith(expected_file_suffix)), None)

                if not target_csv_in_zip:
                    print(f"   ⚠️ Доступні файли в архіві: {csv_files}")
                    raise Exception(f"Файл для {TARGET_YEAR} року не знайдено в архіві!")

                print(f"   🎯 Знайдено цільовий файл: '{target_csv_in_zip}'")

                tmp_csv_path = CSV_FILENAME + ".tmp"

                try:
                    print(f"   ⚙️ Витягуємо '{target_csv_in_zip}' атомарно...")
                    with zip_ref.open(target_csv_in_zip) as source, open(tmp_csv_path, "wb") as target:
                        shutil.copyfileobj(source, target)

                    if os.path.exists(CSV_FILENAME):
                        os.remove(CSV_FILENAME)
                    os.rename(tmp_csv_path, CSV_FILENAME)
                    print(f"✅ Успіх! Файл '{CSV_FILENAME}' (дані {TARGET_YEAR} року) збережено безпечно.")

                except PermissionError:
                    raise Exception(f"Файл {CSV_FILENAME} заблоковано іншою програмою. Закрийте Excel або інші скрипти.")
                except Exception as extract_err:
                    raise Exception(f"Помилка фізичного запису на диск: {extract_err}")
                finally:
                    if os.path.exists(tmp_csv_path):
                        os.remove(tmp_csv_path)

        except Exception as e:
            print(f"❌ Системна помилка під час роботи з архівом: {e}")
    else:
        print("❌ Критична помилка: Архів досі пошкоджений.")
else:
    print("❌ Помилка: Архів не знайдено. Перезапустіть попередній блок завантаження.")

📦 Аналізуємо вміст архіву 'WorldHappinessDataSet/world-happiness.zip'...
   🎯 Знайдено цільовий файл: '2017.csv'
   ⚙️ Витягуємо '2017.csv' атомарно...
✅ Успіх! Файл 'WorldHappinessDataSet/2017.csv' (дані 2017 року) збережено безпечно.


**4. Прочитати дані та відобразити загальну інформацію:**

In [59]:
print(f"📂 Завантаження набору даних з файлу: {CSV_FILENAME}\n")
df = pd.read_csv(CSV_FILENAME)

print("Красивий Вивід - Перші 5 рядків набору даних:")
display(df.head().style.background_gradient(cmap='Blues').set_properties(**TABLE_PROPS))

print("Технічний Вивід:\nОписові статистики:")
display(df.describe().T.style.background_gradient(cmap=DESCRIBE_CMAP).format("{:.4f}"))

print("Інформація про типи ознак та пропуски:")
df.info()

📂 Завантаження набору даних з файлу: WorldHappinessDataSet/2017.csv

Красивий Вивід - Перші 5 рядків набору даних:


,Country,Happiness.Rank,Happiness.Score,Whisker.high,Whisker.low,Economy..GDP.per.Capita.,Family,Health..Life.Expectancy.,Freedom,Generosity,Trust..Government.Corruption.,Dystopia.Residual
0,Norway,1,7.537000,7.594445,7.479556,1.616463,1.533524,0.796667,0.635423,0.362012,0.315964,2.277027
1,Denmark,2,7.522000,7.581728,7.462272,1.482383,1.551122,0.792566,0.626007,0.355280,0.400770,2.313707
2,Iceland,3,7.504000,7.622030,7.385970,1.480633,1.610574,0.833552,0.627163,0.475540,0.153527,2.322715
3,Switzerland,4,7.494000,7.561772,7.426227,1.564980,1.516912,0.858131,0.620071,0.290549,0.367007,2.276716
4,Finland,5,7.469000,7.527542,7.410458,1.443572,1.540247,0.809158,0.617951,0.245483,0.382612,2.430182


Технічний Вивід:
Описові статистики:


,count,mean,std,min,25%,50%,75%,max
Happiness.Rank,155.0000,78.0000,44.8888,1.0000,39.5000,78.0000,116.5000,155.0000
Happiness.Score,155.0000,5.3540,1.1312,2.6930,4.5055,5.2790,6.1015,7.5370
Whisker.high,155.0000,5.4523,1.1185,2.8649,4.6082,5.3700,6.1946,7.6220
Whisker.low,155.0000,5.2557,1.1450,2.5211,4.3750,5.1932,6.0065,7.4796
Economy..GDP.per.Capita.,155.0000,0.9847,0.4208,0.0000,0.6634,1.0646,1.3180,1.8708
Family,155.0000,1.1889,0.2873,0.0000,1.0426,1.2539,1.4143,1.6106
Health..Life.Expectancy.,155.0000,0.5513,0.2371,0.0000,0.3699,0.6060,0.7230,0.9495
Freedom,155.0000,0.4088,0.1500,0.0000,0.3037,0.4375,0.5166,0.6582
Generosity,155.0000,0.2469,0.1348,0.0000,0.1541,0.2315,0.3238,0.8381
Trust..Government.Corruption.,155.0000,0.1231,0.1017,0.0000,0.0573,0.0898,0.1533,0.4643


Інформація про типи ознак та пропуски:
<class 'pandas.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        155 non-null    str    
 1   Happiness.Rank                 155 non-null    int64  
 2   Happiness.Score                155 non-null    float64
 3   Whisker.high                   155 non-null    float64
 4   Whisker.low                    155 non-null    float64
 5   Economy..GDP.per.Capita.       155 non-null    float64
 6   Family                         155 non-null    float64
 7   Health..Life.Expectancy.       155 non-null    float64
 8   Freedom                        155 non-null    float64
 9   Generosity                     155 non-null    float64
 10  Trust..Government.Corruption.  155 non-null    float64
 11  Dystopia.Residual              155 non-null    float64
dtypes: float64(10), int64(

**5. Побудувати діаграми розподілу числових ознак:**

**6. Відібрати числових ознак та кореляційну матрицю:**

**7. Зробити висновок:**

**8. Відобразити розподіл:**

**9. Застосувати стандартизацію даних:**

**10. Відобразити статистики:**

**11. Побудувати модель кластеризації:**

**12. Побудувати теплову мапу:**

**13. Дослідити вплив:**

**13.5.\*\* Експорт навченої моделі ШІ:**

**13.9.\*\* Класифікація щастя:**

**14. Висновок:**